In [1]:
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

from IPPy.utilities.metrics import PSNR, SSIM

from utils.data_utils import load_normalized_image
from IPPy.nn.models import UNet


# Input
TEST_DIR = Path("../dataset_resized/test_resized")
UNET_DIR = Path("../modelli_unet")

# Confermato da 04_preprocessing.ipynb: l'input della UNet e' la ricostruzione FBP
# (il target di training era invece la ricostruzione TV, non la ground truth)
UNET_INPUT_DIR = Path("../fbp")
UNET_RECON_DIR = Path("../unet_reconstructions")

PATIENT = "C081"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configurazioni Limited-Angle CT
N_ANGLES_CONFIGS = [90, 45, 30, 15]

ANGLE_INTERVALS = {
    90: "[-45°, 45°]",
    45: "[-45°, 45°]",
    30: "(-30°, 30°)",
    15: "(-30°, 30°)",
}

GT_DIR = TEST_DIR / PATIENT
UNET_INPUT_TEST_DIR = UNET_INPUT_DIR / "test"
UNET_TEST_DIR = UNET_RECON_DIR / "test"

In [ ]:
# Caricamento del modello UNet allenato per una data configurazione angolare
# I pesi sono un OrderedDict puro (state_dict), senza metadata di epoca/val_loss:

def infer_unet_config(state_dict):
    """Ricava i parametri costruttivi di IPPy.nn.models.UNet dalle shape del checkpoint,
    cosi' non serve indovinare middle_ch a mano."""

    ch_in = state_dict["preprocess.conv.0.weight"].shape[1]
    middle_ch = [state_dict["preprocess.conv.0.weight"].shape[0]]

    i = 0
    while f"encoder_layers.{i}.conv.conv2.weight" in state_dict:
        middle_ch.append(state_dict[f"encoder_layers.{i}.conv.conv2.weight"].shape[0])
        i += 1
    n_blocks = i

    ch_out = state_dict["postprocess.weight"].shape[0]

    return {
        "ch_in": ch_in,
        "ch_out": ch_out,
        "middle_ch": middle_ch,
        "n_layers_per_block": 2,  # confermato dal pattern conv.{0,1,3,4} nel checkpoint
        "down_layers": tuple(["ResDownBlock"] * n_blocks),
        "up_layers": tuple(["ResUpBlock"] * n_blocks),
    }


def load_unet_model(n_angles: int) -> torch.nn.Module:

    weights_path = UNET_DIR / str(n_angles) / "model_weights.pth"
    state_dict = torch.load(weights_path, map_location=DEVICE)

    config = infer_unet_config(state_dict)
    print(f"{n_angles} angoli -> {config}")

    model = UNet(**config)
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()

    return model


models = {n_angles: load_unet_model(n_angles) for n_angles in N_ANGLES_CONFIGS}

90 angoli -> {'ch_in': 1, 'ch_out': 1, 'middle_ch': [32, 64, 128, 256], 'n_layers_per_block': 2, 'down_layers': ('ResDownBlock', 'ResDownBlock', 'ResDownBlock'), 'up_layers': ('ResUpBlock', 'ResUpBlock', 'ResUpBlock')}
45 angoli -> {'ch_in': 1, 'ch_out': 1, 'middle_ch': [32, 64, 128, 256], 'n_layers_per_block': 2, 'down_layers': ('ResDownBlock', 'ResDownBlock', 'ResDownBlock'), 'up_layers': ('ResUpBlock', 'ResUpBlock', 'ResUpBlock')}
30 angoli -> {'ch_in': 1, 'ch_out': 1, 'middle_ch': [32, 64, 128, 256], 'n_layers_per_block': 2, 'down_layers': ('ResDownBlock', 'ResDownBlock', 'ResDownBlock'), 'up_layers': ('ResUpBlock', 'ResUpBlock', 'ResUpBlock')}
15 angoli -> {'ch_in': 1, 'ch_out': 1, 'middle_ch': [32, 64, 128, 256], 'n_layers_per_block': 2, 'down_layers': ('ResDownBlock', 'ResDownBlock', 'ResDownBlock'), 'up_layers': ('ResUpBlock', 'ResUpBlock', 'ResUpBlock')}


In [3]:
# Inferenza della UNet sul test set per ciascuna configurazione angolare
# Le ricostruzioni vengono salvate su disco, con la stessa struttura di tv_reconstructions

for n_angles in N_ANGLES_CONFIGS:

    model = models[n_angles]

    input_paths = sorted(
        (UNET_INPUT_TEST_DIR / str(n_angles) / PATIENT).glob("*.npy"),
        key=lambda p: int(p.stem)
    )

    out_dir = UNET_TEST_DIR / str(n_angles) / PATIENT
    out_dir.mkdir(parents=True, exist_ok=True)

    for input_path in tqdm(input_paths, desc=f"Inferenza UNet [{n_angles} angoli]"):

        x_input = np.load(input_path).astype(np.float32)
        x_input = torch.from_numpy(x_input).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            x_output = model(x_input)

        x_output = x_output.squeeze().cpu().numpy()

        np.save(out_dir / input_path.name, x_output)

Inferenza UNet [90 angoli]: 0it [00:00, ?it/s]
Inferenza UNet [45 angoli]: 0it [00:00, ?it/s]
Inferenza UNet [30 angoli]: 0it [00:00, ?it/s]
Inferenza UNet [15 angoli]: 0it [00:00, ?it/s]


In [4]:
# Controllo corrispondenza e dtype/range delle ricostruzioni UNet sul test set

gt_paths = sorted(
    GT_DIR.glob("*.png"),
    key=lambda p: int(p.stem)
)

print(f"Ground truth {PATIENT}: {len(gt_paths)} slice")


for n_angles in N_ANGLES_CONFIGS:

    unet_paths = sorted(
        (UNET_TEST_DIR / str(n_angles) / PATIENT).glob("*.npy"),
        key=lambda p: int(p.stem)
    )

    gt_stems = {p.stem for p in gt_paths}
    unet_stems = {p.stem for p in unet_paths}

    missing_gt = unet_stems - gt_stems
    missing_unet = gt_stems - unet_stems

    global_min = np.inf
    global_max = -np.inf
    pixels_below_0 = 0
    pixels_above_1 = 0
    total_pixels = 0

    for unet_path in unet_paths:
        image = np.load(unet_path)
        global_min = min(global_min, image.min())
        global_max = max(global_max, image.max())
        pixels_below_0 += np.sum(image < 0)
        pixels_above_1 += np.sum(image > 1)
        total_pixels += image.size

    print(f"\n{n_angles} angoli {ANGLE_INTERVALS[n_angles]}")
    print(f"Ricostruzioni UNet: {len(unet_paths)}")
    print(f"UNet senza ground truth: {len(missing_gt)}")
    print(f"Ground truth senza UNet: {len(missing_unet)}")
    print(f"Range: [{global_min:.4f}, {global_max:.4f}]")
    print(f"Pixel sotto 0: {100 * pixels_below_0 / total_pixels:.4f}%")
    print(f"Pixel sopra 1: {100 * pixels_above_1 / total_pixels:.4f}%")

Ground truth C081: 327 slice

90 angoli [-45°, 45°]
Ricostruzioni UNet: 0
UNet senza ground truth: 0
Ground truth senza UNet: 327
Range: [inf, -inf]


ZeroDivisionError: division by zero

In [ ]:
# Caricamento delle ricostruzioni UNet con clipping nel range [0,1]

def load_unet_reconstruction(path: Path):
    image = np.load(path).astype(np.float32)
    return np.clip(image, 0.0, 1.0)

In [ ]:
# Calcolo PSNR e SSIM tra ricostruzioni UNet e ground truth sul test set

results_unet = {}


for n_angles in N_ANGLES_CONFIGS:

    unet_paths = sorted(
        (UNET_TEST_DIR / str(n_angles) / PATIENT).glob("*.npy"),
        key=lambda p: int(p.stem)
    )

    psnr_values = []
    ssim_values = []
    slice_names = []

    for unet_path in tqdm(unet_paths, desc=f"Metriche UNet [{n_angles} angoli]"):

        gt_path = GT_DIR / f"{unet_path.stem}.png"

        if not gt_path.exists():
            print(f"Ground truth non trovata per {unet_path.name}")
            continue

        gt = load_normalized_image(gt_path)
        unet = load_unet_reconstruction(unet_path)

        if gt.shape != unet.shape:
            print(f"Shape diverse per slice {unet_path.stem}: GT={gt.shape}, UNet={unet.shape}")
            continue

        x_true = torch.from_numpy(gt).float().unsqueeze(0).unsqueeze(0)
        x_unet = torch.from_numpy(unet).float().unsqueeze(0).unsqueeze(0)

        psnr_value = float(PSNR(x_unet, x_true))
        ssim_value = float(SSIM(x_unet, x_true))

        psnr_values.append(psnr_value)
        ssim_values.append(ssim_value)
        slice_names.append(unet_path.stem)

    results_unet[n_angles] = {
        "slice": slice_names,
        "PSNR": psnr_values,
        "SSIM": ssim_values,
    }

In [ ]:
# Risultati medi di PSNR e SSIM per ciascuna configurazione angolare

print("RISULTATI UNET SUL TEST SET - PAZIENTE " + PATIENT)

for n_angles in N_ANGLES_CONFIGS:

    psnr_values = np.array(results_unet[n_angles]["PSNR"])
    ssim_values = np.array(results_unet[n_angles]["SSIM"])

    print(f"\n{n_angles} angoli {ANGLE_INTERVALS[n_angles]}")
    print(f"Numero slice: {len(psnr_values)}")
    print(f"PSNR medio: {psnr_values.mean():.2f} ± {psnr_values.std():.2f} dB")
    print(f"SSIM medio: {ssim_values.mean():.4f} ± {ssim_values.std():.4f}")

In [ ]:
# Grafico PSNR e SSIM medi per configurazione angolare (UNet, con overlay TV opzionale)

psnr_means_unet = [np.mean(results_unet[n_angles]["PSNR"]) for n_angles in N_ANGLES_CONFIGS]
ssim_means_unet = [np.mean(results_unet[n_angles]["SSIM"]) for n_angles in N_ANGLES_CONFIGS]

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(N_ANGLES_CONFIGS, psnr_means_unet, marker="o", label="UNet")
# axs[0].plot(N_ANGLES_CONFIGS, psnr_means_tv, marker="o", label="TV")
axs[0].set_xlabel("Numero di angoli")
axs[0].set_ylabel("PSNR medio (dB)")
axs[0].set_title("UNet - PSNR sul test set")
axs[0].set_xticks(N_ANGLES_CONFIGS)
axs[0].grid(alpha=0.3)
axs[0].legend()

axs[1].plot(N_ANGLES_CONFIGS, ssim_means_unet, marker="o", label="UNet")
# axs[1].plot(N_ANGLES_CONFIGS, ssim_means_tv, marker="o", label="TV")
axs[1].set_xlabel("Numero di angoli")
axs[1].set_ylabel("SSIM medio")
axs[1].set_title("UNet - SSIM sul test set")
axs[1].set_xticks(N_ANGLES_CONFIGS)
axs[1].grid(alpha=0.3)
axs[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Confronto visivo tra ground truth e ricostruzioni UNet

example_gt_path = gt_paths[len(gt_paths) // 2]
slice_name = example_gt_path.stem

gt = load_normalized_image(example_gt_path)


fig, axs = plt.subplots(1, 5, figsize=(20, 4))


axs[0].imshow(gt, cmap="gray", vmin=0, vmax=1)
axs[0].set_title("Ground Truth")
axs[0].axis("off")


for ax, n_angles in zip(axs[1:], N_ANGLES_CONFIGS):

    unet_path = (
        UNET_TEST_DIR
        / str(n_angles)
        / PATIENT
        / f"{slice_name}.npy"
    )

    unet = load_unet_reconstruction(unet_path)

    x_true = torch.from_numpy(gt).float().unsqueeze(0).unsqueeze(0)
    x_unet = torch.from_numpy(unet).float().unsqueeze(0).unsqueeze(0)

    psnr_value = float(PSNR(x_unet, x_true))
    ssim_value = float(SSIM(x_unet, x_true))

    ax.imshow(unet, cmap="gray", vmin=0, vmax=1)

    ax.set_title(
        f"{n_angles} angoli\n"
        f"PSNR: {psnr_value:.2f} dB\n"
        f"SSIM: {ssim_value:.4f}"
    )

    ax.axis("off")


plt.tight_layout()
plt.show()